# Quark: compare components on Google Colab

Trains every **arm** of a comparison once per **seed** on the same token budget and reports how each arm differs from the reference, with paired seeds to cancel run-to-run noise (plan §18.2). Data, runs and the report live on **Google Drive**.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Edit the **Settings** cell (the spec), then *Runtime → Run all*.

**Interrupted?** Just *Run all* again: finished runs are skipped and an unfinished run resumes from its last checkpoint. The compare also stops on its own ~20 minutes before the 12-hour session limit.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/quark")  # models, checkpoints and cached data live here
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE_ROOT)

## 3. Install Quark

In [ ]:
QUARK_REF = "main"  # a branch, tag or commit of the quark repo
%pip install -q "git+https://github.com/cpalmquist19/quark.git@{QUARK_REF}"

In [ ]:
import sys
import torch
import quark
from quark.hardware import detect_profile, load_profile

print(f"python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | quark {quark.__version__}")
assert torch.cuda.is_available(), "No GPU visible to PyTorch." + (
    " Runtime -> Change runtime type -> T4 GPU." if "google.colab" in sys.modules
    else " Is the notebook using the project's .venv kernel (CUDA build of torch)?")

hw = detect_profile()
print(f"GPU: {hw.gpu_name} | {hw.vram_gb:g} GB ({hw.usable_vram_gb:g} usable) | precision {hw.precision} | SDPA {hw.sdpa_kernels}")
TARGET = "colab-t4" if "google.colab" in sys.modules else "rtx3070ti-laptop"
if hw.compute_capability != load_profile(TARGET).compute_capability:
    print(f"warning: this GPU differs from the expected target ({TARGET}); settings may need adjusting")

## Settings
The experiment spec. Each arm is a partial override of the base config (`{}` = the base itself); the first arm is the reference unless `reference` is set. `data` is what every arm trains on. Change `name` for a new experiment: results go to `MyDrive/quark/experiments/<name>/`.

In [ ]:
EXPERIMENTS_ROOT = DRIVE_ROOT / "experiments"
DATA_ROOT = DRIVE_ROOT / "data"          # tokenized data is cached on Drive, then copied to local disk
LOCAL_DATA_ROOT = Path("/content/data")

SPEC_YAML = '''
# First component comparison on the ablation-s rung (plan §18.2).
#   quark compare check experiments/first-batch.yaml
#   quark compare run experiments/first-batch.yaml --data <prepared data folder>
# On Colab, use notebooks/quark_compare_colab.ipynb.
#
# Each arm overrides the base config; {} is the base itself. All arms train once per seed on the same
# token budget, and every arm is compared with the reference (the first arm unless `reference` is set).
# Arms take ~10-12 min each on a T4 at ablation-s (estimate): 8 arms x 2 seeds is roughly 3 hours.

name: first-batch
base: ablation-s
seeds: [1, 2]
# What every arm trains on (DataConfig fields). FineWeb-Edu, the base of the target mix (plan §18): which
# components win depends on the kind of text, not on the exact domain weights, so these results should
# carry over to the full mix. 500M tokens are streamed once (~2 GB) and cached on Drive.
data:
  source: HuggingFaceFW/fineweb-edu
  subset: sample-10BT
  token_limit: 500000000
  tokenizer: {type: train, size: 8192}
arms:
  baseline: {}                                   # RoPE, MHA, RMSNorm, SwiGLU, tied head, AdamW, cosine
  muon:
    train: {optimizer: muon}
  muon-lr-half:                                  # Muon gets its own small LR sweep (plan §18.2)
    train: {optimizer: muon, lr: 1.1e-3}
  wsd:
    train: {schedule: wsd}
  qk-norm:
    model: {toggles: {qk_norm: true}}
  z-loss:
    model: {toggles: {z_loss: true}}
  alibi:
    model: {slots: {position: alibi}}
  relu2:
    model: {slots: {mlp: relu2}}
'''

## Check the arms
Every arm is validated before any GPU time is spent; invalid combinations stop here with the reason.

In [ ]:
import yaml
from quark.compare import CompareSpec, arm_config, validate
from quark.model import count_params

spec = CompareSpec.model_validate(yaml.safe_load(SPEC_YAML))
EXP_DIR = EXPERIMENTS_ROOT / spec.name
issues = validate(spec, hw)
print(f"{spec.name}: {len(spec.arms)} arms x {len(spec.seeds)} seeds = {len(spec.arms) * len(spec.seeds)} runs -> {EXP_DIR}")
for arm in spec.arms:
    arm_cfg = arm_config(spec, arm, spec.seeds[0])
    notes = "; ".join(f"{i.severity}: {i.message}" for i in issues[arm])
    tag = " (reference)" if arm == spec.ref else ""
    print(f"  {arm + tag:24s} {count_params(arm_cfg.model) / 1e6:6.2f}M params  {arm_cfg.train.max_tokens / 1e6:,.0f}M tokens  {notes}")
cfg = arm_config(spec, spec.ref, spec.seeds[0])  # the data settings every arm shares

## Data
Tokenized once, then reused by every arm and every later experiment on the same data.

In [ ]:
import shutil
from quark.data import default_data_dir, prepare_dataset

DRIVE_DATA_DIR = default_data_dir(cfg.data, root=DATA_ROOT)
DATA_DIR = LOCAL_DATA_ROOT / DRIVE_DATA_DIR.name  # local copy: much faster random reads than Drive
if not (DRIVE_DATA_DIR / "meta.json").exists():
    if cfg.data.mix:
        raise FileNotFoundError(f"mix {cfg.data.mix!r} isn't prepared on Drive yet ({DRIVE_DATA_DIR}); "
                                "build it with notebooks/quark_data_colab.ipynb first")
    print(f"preparing {cfg.data.source} -> {DATA_DIR} (then caching to Drive) ...")
    prepare_dataset(cfg.data, DATA_DIR, progress=lambda d, n: d % 100_000 == 0 and print(f"  {d:,} docs, {n / 1e6:,.0f}M tokens"))
    shutil.copytree(DATA_DIR, DRIVE_DATA_DIR, dirs_exist_ok=True)
elif not (DATA_DIR / "meta.json").exists():
    print(f"copying cached data from Drive to {DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, DATA_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("raw"))

In [ ]:
from quark.data import MixData, load_data

data = load_data(DATA_DIR)
meta = data.meta
if isinstance(data, MixData):
    weights = data.resolve_weights(cfg.data.weights)
    print(f"mix {meta['name']} | vocab {meta['vocab_size']:,}")
    for name, src in meta["sources"].items():
        w = weights.get(name, 0.0)
        epochs = w * cfg.train.max_tokens / max(1, src["train_tokens"])
        print(f"  {name:18s} train {src['train_tokens'] / 1e6:7,.1f}M | val {src['val_tokens'] / 1e6:5,.2f}M | "
              f"weight {w:4.0%} | {epochs:.2f} epochs planned")
    sample_tokens = next(iter(data.sources.values())).val
else:
    print(f"vocab {meta['vocab_size']:,} | train {meta['train_tokens'] / 1e6:,.1f}M tokens ({meta['train_docs']:,} docs) "
          f"| val {meta['val_tokens'] / 1e6:,.2f}M tokens ({meta['val_docs']:,} docs)")
    epochs = cfg.train.max_tokens / meta["train_tokens"]
    print(f"planned training: {epochs:.2f} epochs" + ("  (multi-epoch: watch for overfitting)" if epochs > 1 else ""))
    sample_tokens = data.val
print("\nsample (decoded from val):\n" + data.tokenizer().decode(sample_tokens[:150].tolist()))

## Time estimate
One real training step of the reference arm. Slower arms (e.g. Muon) take longer.

In [ ]:
from quark.train import dry_run

done = len(list(EXP_DIR.glob("runs/*/*/result.json")))
total = len(spec.arms) * len(spec.seeds)
check = dry_run(cfg, DATA_DIR)
print(f"reference arm: {check['tokens_per_sec'] / 1e3:.1f}k tokens/s, peak GPU memory {check['peak_mem_gb']:.2f} GB "
      f"of {hw.usable_vram_gb:g} usable | ~{check['est_hours'] * 60:.0f} min per run")
print(f"{total - done} of {total} runs left: ~{check['est_hours'] * (total - done):.1f} h")

## Run
Runs go seed by seed (every arm for the first seed, then the next), so an interrupted compare still has balanced results. Each run shows its own dashboard; the report updates after every run.

In [ ]:
from quark.compare import run_compare

summary = run_compare(spec, DATA_DIR, EXP_DIR, hardware=hw, live_plot=True)

## Report
Δ < 0 means the arm beat the reference. The report explains the verdicts; `report.md`, `results.json` and `curves.png` are in the experiment folder on Drive.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((EXP_DIR / "report.md").read_text(encoding="utf-8")))
if (EXP_DIR / "curves.png").exists():
    display(Image(filename=str(EXP_DIR / "curves.png")))